# DBSCAN-Based Clustering of Multi-Source Flash-Flood Precursor Data

**Study area:** Upper Beas Basin, Kullu district, Himachal Pradesh, India  
**Study period:** 2015–2025  
**Institution:** Lovely Professional University

## Research notebook

This notebook documents the final analytical workflow for the Upper Beas study. It is organized as a reproducible sequence rather than a record of exploratory work. Temporary experiments, dashboard-editing commands, repeated diagnostics and authentication outputs from the working notebook have been excluded.

The analysis integrates daily CHIRPS rainfall, NASA SMAP soil moisture, terrain and grid attributes, and a historical flood-event inventory. DBSCAN is used to identify dense environmental-spatial patterns. Historical incidents are kept separate from model features and are used only for post hoc comparison.

> **Interpretation boundary:** This is an exploratory clustering study. The analysis does not establish an operational flood forecast or warning system. Historical event validation is limited by the small number of dated and geolocated incidents.

## 1. Workflow and reproducibility

The notebook expects the processed input datasets produced by the project's data-preparation workflow. It does **not** repeat the Google Earth Engine exports or download satellite products automatically. This avoids unexpectedly long downloads and keeps the analytical steps separate from data acquisition.

Recommended repository layout:

```text
project/
├── data/
│   └── processed/
├── notebooks/
│   └── Upper_Beas_Flood_Research.ipynb
├── visualizations/
└── reports/
```

Place the required processed CSV files in `data/processed/`. The code uses paths relative to this repository, not a personal Windows directory. Run the cells from top to bottom. The full-period DBSCAN and parameter-sensitivity sections are computationally intensive because clustering is performed separately for each date.

In [ ]:
from pathlib import Path
import sys
import hashlib
import platform
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import pyproj

from pyproj import Transformer
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

# Run Jupyter from the repository root, or open this notebook from its notebooks/ folder.
CURRENT_DIR = Path.cwd().resolve()
if (CURRENT_DIR / "data" / "processed").is_dir():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.name == "notebooks" and (CURRENT_DIR.parent / "data" / "processed").is_dir()):
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise FileNotFoundError(
        "Could not locate the project root. Start Jupyter in the repository root "
        "or place this notebook in the repository's notebooks/ folder."
    )

DATA_DIR = PROJECT_ROOT / "data" / "processed"
FIGURE_DIR = PROJECT_ROOT / "visualizations"
OUTPUT_DIR = DATA_DIR
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("pyproj:", pyproj.__version__)
print("Project root:", PROJECT_ROOT)
print("Processed data directory:", DATA_DIR)
print("Figures directory:", FIGURE_DIR)


## 2. Input data and provenance

The analytical notebook requires these prepared inputs:

| File | Purpose |
|---|---|
| `integrated_environmental_dataset_2015_2025.csv` | Daily rainfall, SMAP soil moisture and terrain attributes for each grid cell |
| `grid_master_325.csv` | The 325-cell spatial grid and centroid coordinates |
| `historical_flood_events_final.csv` | Cleaned historical flood inventory |
| `historical_events_grid_matched.csv` | Historical events associated with nearest grid cells |
| `spatially_matched_event_control_results.csv` | Event/control pairs used for the spatially matched comparison |

**Scope:** this notebook reproduces the analytical stage from the integrated environmental table onward. It does not recreate Google Earth Engine extraction, HydroBASINS boundary construction, grid generation, source-data cleaning or the event/control matching procedure. Those prepared files and their provenance must be archived separately. The repository should include a data dictionary, source citations, access dates, processing notes and checksums for the exact input files. The large input datasets are not bundled here.

Do not upload Google Earth Engine credentials, private keys or authentication files. Check the terms of use for every source dataset before redistributing it.


In [ ]:
required_files = [
    "integrated_environmental_dataset_2015_2025.csv",
    "grid_master_325.csv",
    "historical_flood_events_final.csv",
    "historical_events_grid_matched.csv",
    "spatially_matched_event_control_results.csv",
]

missing_files = [name for name in required_files if not (DATA_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(
        "Required input file(s) are missing from data/processed:\n- "
        + "\n- ".join(missing_files)
    )

# Record SHA-256 hashes so another researcher can verify that they used identical inputs.
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

input_manifest = []
for name in required_files:
    path = DATA_DIR / name
    input_manifest.append({
        "file": name,
        "size_mb": round(path.stat().st_size / (1024**2), 3),
        "sha256": sha256_file(path)
    })

input_manifest = pd.DataFrame(input_manifest)
display(input_manifest)
input_manifest.to_csv(OUTPUT_DIR / "input_data_manifest_sha256.csv", index=False)


## 3. Study area and spatial grid

The working catchment boundary was assembled from five HydroBASINS Level 8 units upstream of the Pandoh outlet. The selected unit identifiers are `4080712380`, `4080712390`, `4080712400`, `4080707380` and `4080707370`. The project records the combined area as approximately 5,657 km². This boundary is treated as a **provisional research boundary**, not an officially surveyed basin delineation.

The catchment was represented by 325 nominal 5 km × 5 km grid cells. Centroid coordinates are stored in geographic coordinates (WGS84); projected coordinates are calculated in UTM zone 43N (EPSG:32643) for spatial distance calculations.

In [ ]:
grid = pd.read_csv(DATA_DIR / "grid_master_325.csv")

required_grid_columns = {"grid_id", "longitude", "latitude"}
if not required_grid_columns.issubset(grid.columns):
    raise ValueError(f"Grid file must contain {sorted(required_grid_columns)}")
if grid["grid_id"].isna().any() or grid["grid_id"].duplicated().any():
    raise ValueError("Grid IDs must be non-missing and unique")
if len(grid) != 325:
    raise ValueError(f"Expected 325 grid cells; found {len(grid)}")
if not grid[["longitude", "latitude"]].apply(np.isfinite).all().all():
    raise ValueError("Grid coordinates contain missing or non-finite values")

transformer = Transformer.from_crs("EPSG:4326", "EPSG:32643", always_xy=True)
grid["x_utm"], grid["y_utm"] = transformer.transform(
    grid["longitude"].to_numpy(), grid["latitude"].to_numpy()
)
grid = grid.sort_values("grid_id", kind="stable").reset_index(drop=True)

print("Grid cells:", grid["grid_id"].nunique())
print("Grid coordinate bounds:")
display(grid[["longitude", "latitude", "x_utm", "y_utm"]].describe().round(3))


In [ ]:
study_map = FIGURE_DIR / "01_upper_beas_study_area_map.png"
if study_map.exists():
    from IPython.display import display, Image
    display(Image(filename=str(study_map)))
else:
    print("Study-area figure not found:", study_map)

## 4. Load and inspect the integrated environmental dataset

The integrated table contains one record per date and grid cell. CHIRPS daily precipitation is expressed in mm/day. SMAP surface and root-zone soil-moisture fields are retained as supplied by the preprocessing workflow. Missing SMAP observations are not replaced with zero; the satellite product's temporal availability is explicitly represented.

In [ ]:
environment = pd.read_csv(
    DATA_DIR / "integrated_environmental_dataset_2015_2025.csv",
    parse_dates=["date"]
)

expected_columns = {
    "date", "grid_id", "rainfall_mm", "sm_surface", "sm_rootzone",
    "elevation_mean_m", "slope_mean_deg"
}
missing_columns = expected_columns - set(environment.columns)
if missing_columns:
    raise ValueError(f"Integrated dataset is missing columns: {sorted(missing_columns)}")
if environment[["date", "grid_id"]].isna().any().any():
    raise ValueError("Integrated dataset contains missing date/grid_id values")
if environment.duplicated(["date", "grid_id"]).any():
    raise ValueError("Integrated dataset contains duplicate date-grid records")
if set(environment["grid_id"].unique()) != set(grid["grid_id"].unique()):
    raise ValueError("Integrated dataset grid IDs do not exactly match grid_master_325.csv")

print("Rows:", len(environment))
print("Columns:", len(environment.columns))
print("Date range:", environment["date"].min(), "to", environment["date"].max())
print("Unique dates:", environment["date"].nunique())
print("Unique grid cells:", environment["grid_id"].nunique())
print("Rows per date (expected 325):")
display(environment.groupby("date").size().value_counts().sort_index().to_frame("number_of_dates"))
if not (environment.groupby("date").size() == 325).all():
    raise ValueError("Every integrated date must contain exactly 325 grid-cell records")

display(environment[[
    "rainfall_mm", "sm_surface", "sm_rootzone",
    "elevation_mean_m", "slope_mean_deg"
]].describe().round(3))
if "smap_available" in environment.columns:
    print("\nSMAP availability:")
    display(environment["smap_available"].value_counts(dropna=False))


## 5. Antecedent rainfall and lagged soil-moisture features

To avoid using rainfall from the day being clustered as part of its own antecedent history, the 3-, 7- and 15-day rainfall sums use `shift(1)` before the rolling window. Surface and root-zone soil moisture are also shifted by one day within each grid cell. This makes the environmental inputs represent conditions available before the target date.

The first days of each grid-cell time series will consequently have missing antecedent values. These are retained as missing at this stage and excluded from the modelling table later; they are not filled with zero.

In [ ]:
environment = environment.sort_values(["grid_id", "date"]).reset_index(drop=True)
grouped_rain = environment.groupby("grid_id")["rainfall_mm"]

environment["rainfall_3day"] = grouped_rain.transform(
    lambda values: values.shift(1).rolling(3, min_periods=3).sum()
)
environment["rainfall_7day"] = grouped_rain.transform(
    lambda values: values.shift(1).rolling(7, min_periods=7).sum()
)
environment["rainfall_15day"] = grouped_rain.transform(
    lambda values: values.shift(1).rolling(15, min_periods=15).sum()
)

environment["sm_surface_previous_day"] = (
    environment.groupby("grid_id")["sm_surface"].shift(1)
)
environment["sm_rootzone_previous_day"] = (
    environment.groupby("grid_id")["sm_rootzone"].shift(1)
)

antecedent_columns = [
    "rainfall_3day", "rainfall_7day", "rainfall_15day",
    "sm_surface_previous_day", "sm_rootzone_previous_day"
]
print("Missing antecedent/lagged features:")
display(environment[antecedent_columns].isna().sum().to_frame("missing_count"))

## 6. Prepare the modelling observations

The model uses observations for which all five antecedent/lagged environmental features are available. Rainfall features are transformed with `log1p` to reduce skew. Feature scaling is calibrated on the period before 1 January 2018 and then applied to the complete modelling period. This preserves a fixed historical calibration rather than refitting the scaler independently on every date.

The model features are:

- log-transformed 3-, 7- and 15-day antecedent rainfall;
- previous-day SMAP surface soil moisture;
- previous-day SMAP root-zone soil moisture.

Spatial coordinates are added separately in the DBSCAN distance construction.

In [ ]:
model_df = environment.dropna(subset=antecedent_columns).copy()

for column in ["rainfall_3day", "rainfall_7day", "rainfall_15day"]:
    model_df[column + "_log"] = np.log1p(model_df[column])

raw_features = [
    "rainfall_3day_log", "rainfall_7day_log", "rainfall_15day_log",
    "sm_surface_previous_day", "sm_rootzone_previous_day"
]
scaled_features = [column + "_scaled" for column in raw_features]

calibration = model_df[model_df["date"] < "2018-01-01"]
if calibration.empty:
    raise ValueError("No observations found in the 2015–2017 calibration period")

scaler = StandardScaler()
scaler.fit(calibration[raw_features])
model_df[scaled_features] = scaler.transform(model_df[raw_features])

model_df = model_df.sort_values(["date", "grid_id"], kind="stable").reset_index(drop=True)

model_df = model_df.merge(
    grid[["grid_id", "x_utm", "y_utm"]],
    on="grid_id", how="left", validate="many_to_one"
)

if model_df[["x_utm", "y_utm"]].isna().any().any():
    raise ValueError("Some model observations could not be matched to grid coordinates")

print("Model observations:", len(model_df))
print("Date range:", model_df["date"].min(), "to", model_df["date"].max())
print("Unique dates:", model_df["date"].nunique())
print("Unique grid cells:", model_df["grid_id"].nunique())
print("Calibration observations:", len(calibration))
print("Calibration period:", calibration["date"].min(), "to", calibration["date"].max())

rows_per_model_date = model_df.groupby("date").size()
if not (rows_per_model_date == 325).all():
    raise ValueError("Every model date must contain exactly 325 complete grid-cell observations")

scaler_audit = pd.DataFrame({
    "feature": raw_features,
    "calibration_mean": scaler.mean_,
    "calibration_scale": scaler.scale_
})
scaler_audit.to_csv(OUTPUT_DIR / "feature_scaler_calibration.csv", index=False)

display(model_df[scaled_features].describe().round(3))

## 7. DBSCAN distance construction

For each date, DBSCAN is applied to the 325-cell spatial grid. Projected coordinates are divided by 5,000 so one coordinate unit corresponds to 5 km. The five standardized environmental features are divided by the environmental weight, 0.3, before being combined with the spatial coordinates:

\[
X_i = \left(\frac{x_i}{5000},\frac{y_i}{5000},\frac{z_{i1}}{0.3},\ldots,\frac{z_{i5}}{0.3}\right)
\]

Here, \(z_{ij}\) denotes a scaled environmental feature. The distance threshold `eps` and minimum-neighbour count `min_samples` determine the density definition. A DBSCAN label of `-1` denotes noise; other labels identify clusters. The clustering is unsupervised and does not use historical event labels.


In [ ]:
ENVIRONMENTAL_WEIGHT = 0.3
DBSCAN_FEATURES = scaled_features


def make_daily_matrix(day):
    day = day.sort_values("grid_id", kind="stable")
    environmental = day[DBSCAN_FEATURES].to_numpy(dtype=float)
    spatial = day[["x_utm", "y_utm"]].to_numpy(dtype=float) / 5000.0
    return np.column_stack([spatial, environmental / ENVIRONMENTAL_WEIGHT])


def summarize_labels(labels):
    labels = np.asarray(labels)
    cluster_count = len(set(labels)) - (1 if -1 in labels else 0)
    noise_percent = float((labels == -1).mean() * 100)
    return cluster_count, noise_percent

print("Environmental features:", DBSCAN_FEATURES)
print("Environmental weight:", ENVIRONMENTAL_WEIGHT)
print("Unique grid cells per full date (expected):", grid["grid_id"].nunique())

## 8. Baseline full-period DBSCAN

The baseline configuration is `eps=2.0` and `min_samples=4`. The model is fitted independently for each date. Daily cluster count, noise percentage and clustered coverage are then summarized over the study period. This loop is computationally more expensive than the descriptive cells above.

In [ ]:
BASELINE_EPS = 2.0
BASELINE_MIN_SAMPLES = 4

baseline_rows = []
for date, day in model_df.groupby("date", sort=True):
    X = make_daily_matrix(day)
    labels = DBSCAN(
        eps=BASELINE_EPS,
        min_samples=BASELINE_MIN_SAMPLES
    ).fit_predict(X)
    cluster_count, noise_percent = summarize_labels(labels)
    baseline_rows.append({
        "date": date,
        "cluster_count": cluster_count,
        "noise_percent": noise_percent,
        "clustered_percent": 100.0 - noise_percent
    })

baseline_daily = pd.DataFrame(baseline_rows)
baseline_path = OUTPUT_DIR / "dbscan_full_period_cluster_coverage.csv"
baseline_daily.to_csv(baseline_path, index=False)

print("Daily records:", len(baseline_daily))
print("Mean clusters:", round(baseline_daily["cluster_count"].mean(), 2))
print("Mean noise (%):", round(baseline_daily["noise_percent"].mean(), 2))
print("Mean clustered coverage (%):", round(baseline_daily["clustered_percent"].mean(), 2))
print("Saved:", baseline_path)

## 9. Parameter sensitivity analysis

The sensitivity experiment evaluates 15 combinations: `eps` values of 1.0, 1.25, 1.5, 1.75 and 2.0, crossed with `min_samples` values of 4, 6 and 10. Every configuration is evaluated independently for every date. The resulting table summarizes mean and median cluster counts, noise percentage and clustered coverage.

In [ ]:
EPS_VALUES = [1.0, 1.25, 1.5, 1.75, 2.0]
MIN_SAMPLES_VALUES = [4, 6, 10]
PARAMETERS = [(eps, min_samples) for eps in EPS_VALUES for min_samples in MIN_SAMPLES_VALUES]

sensitivity_rows = []
for date, day in model_df.groupby("date", sort=True):
    X = make_daily_matrix(day)
    for eps, min_samples in PARAMETERS:
        labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(X)
        cluster_count, noise_percent = summarize_labels(labels)
        sensitivity_rows.append({
            "date": date,
            "eps": eps,
            "min_samples": min_samples,
            "cluster_count": cluster_count,
            "noise_percent": noise_percent,
            "clustered_percent": 100.0 - noise_percent
        })

sensitivity_daily = pd.DataFrame(sensitivity_rows)
sensitivity_summary = sensitivity_daily.groupby(
    ["eps", "min_samples"], as_index=False
).agg(
    mean_clusters=("cluster_count", "mean"),
    median_clusters=("cluster_count", "median"),
    mean_noise_percent=("noise_percent", "mean"),
    median_noise_percent=("noise_percent", "median"),
    mean_clustered_percent=("clustered_percent", "mean")
)

sensitivity_path = OUTPUT_DIR / "dbscan_wider_parameter_sensitivity.csv"
sensitivity_summary.to_csv(sensitivity_path, index=False)
sensitivity_daily.to_csv(OUTPUT_DIR / "dbscan_wider_parameter_sensitivity_daily.csv", index=False)
display(sensitivity_summary.round(2))
print("Saved:", sensitivity_path)

## 10. Historical flood inventory and event matching

The historical incident inventory is not used to train DBSCAN. It is used only after clustering to compare the cluster/noise status of incident-associated grid cells with spatially matched control cells. Records without usable dates or coordinates cannot contribute to this event-based comparison.

The event/control comparison file records the matched pairs used in the study. Retaining this file makes the comparison traceable and avoids silently changing the control-selection procedure when the notebook is rerun.

In [ ]:
events = pd.read_csv(
    DATA_DIR / "historical_flood_events_final.csv",
    parse_dates=["event_date"]
)
matched_events = pd.read_csv(
    DATA_DIR / "historical_events_grid_matched.csv",
    parse_dates=["event_date"]
)
matched_controls = pd.read_csv(
    DATA_DIR / "spatially_matched_event_control_results.csv",
    parse_dates=["date"]
)

required_event_columns = {"event_date", "latitude", "longitude"}
required_matched_columns = {"event_date", "nearest_grid_id"}
required_pair_columns = {"date", "event_grid", "control_grid", "event_number"}
for label, frame, required in [
    ("historical_flood_events_final.csv", events, required_event_columns),
    ("historical_events_grid_matched.csv", matched_events, required_matched_columns),
    ("spatially_matched_event_control_results.csv", matched_controls, required_pair_columns),
]:
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{label} is missing required columns: {sorted(missing)}")

study_events = events[
    events["event_date"].between("2015-01-01", "2025-12-31")
].copy()

print("Dated inventory records in study period:", len(study_events))
print("Records with coordinates:", study_events[["latitude", "longitude"]].notna().all(axis=1).sum())
if "event_type_group" in study_events.columns:
    display(study_events["event_type_group"].value_counts(dropna=False).to_frame("records"))

print("\nMatched event records:", len(matched_events))
print("Spatially matched control pairs:", len(matched_controls))
print("Control-pair date range:", matched_controls["date"].min(), "to", matched_controls["date"].max())
display(matched_controls.head())


## 11. Event-versus-control comparison

For each dated incident and its matched control, the relevant daily DBSCAN model is refitted. A location is considered clustered when its DBSCAN label is not `-1`. The comparison is repeated for three configurations used in the final analysis:

- baseline: `eps=2.0`, `min_samples=4`;
- selective: `eps=1.5`, `min_samples=6`;
- strict: `eps=1.5`, `min_samples=10`.

This is an event-associated density comparison, not a conventional forecast evaluation. The number of dated events is small, so estimates and p-values must be interpreted cautiously.

In [ ]:
EVENT_CONFIGS = [(2.0, 4), (1.5, 6), (1.5, 10)]
event_rows = []

for eps, min_samples in EVENT_CONFIGS:
    for date, pairs in matched_controls.groupby("date"):
        date = pd.Timestamp(date)
        day = model_df[model_df["date"] == date].copy().reset_index(drop=True)
        if day.empty:
            continue

        labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(make_daily_matrix(day))
        day["cluster"] = labels

        for _, pair in pairs.iterrows():
            event_row = day[day["grid_id"] == pair["event_grid"]]
            control_row = day[day["grid_id"] == pair["control_grid"]]
            if event_row.empty or control_row.empty:
                continue

            event_detected = int(event_row.iloc[0]["cluster"] != -1)
            control_detected = int(control_row.iloc[0]["cluster"] != -1)
            event_rows.append({
                "eps": eps,
                "min_samples": min_samples,
                "event_number": pair.get("event_number", np.nan),
                "event_detected": event_detected,
                "control_detected": control_detected,
                "difference": event_detected - control_detected
            })

event_comparison = pd.DataFrame(event_rows)
if event_comparison.empty:
    raise ValueError("No event/control pairs could be matched to model dates and grid cells")

event_summary = event_comparison.groupby(["eps", "min_samples"], as_index=False).agg(
    event_detection=("event_detected", "mean"),
    control_detection=("control_detected", "mean"),
    mean_difference=("difference", "mean"),
    event_records=("event_number", "count")
)
for column in ["event_detection", "control_detection", "mean_difference"]:
    event_summary[column] *= 100

display(event_summary.round(2))
event_comparison.to_csv(OUTPUT_DIR / "dbscan_candidate_event_comparison.csv", index=False)
event_summary.to_csv(OUTPUT_DIR / "dbscan_candidate_event_summary.csv", index=False)

## 12. Paired McNemar test

For each configuration, the paired event/control outcomes are summarized in a 2 × 2 table. The exact two-sided McNemar p-value is calculated from the discordant pairs (`event_only` and `control_only`). This tests whether the paired detection proportions differ; it does not establish predictive skill.

In [ ]:
def exact_mcnemar_pvalue(event_only, control_only):
    n = int(event_only + control_only)
    if n == 0:
        return 1.0
    k = min(int(event_only), int(control_only))
    probability = sum(math.comb(n, i) for i in range(k + 1)) / (2 ** n)
    return min(1.0, 2.0 * probability)

mcnemar_rows = []
for (eps, min_samples), group in event_comparison.groupby(["eps", "min_samples"]):
    both = ((group["event_detected"] == 1) & (group["control_detected"] == 1)).sum()
    event_only = ((group["event_detected"] == 1) & (group["control_detected"] == 0)).sum()
    control_only = ((group["event_detected"] == 0) & (group["control_detected"] == 1)).sum()
    neither = ((group["event_detected"] == 0) & (group["control_detected"] == 0)).sum()
    mcnemar_rows.append({
        "eps": eps,
        "min_samples": min_samples,
        "both_clustered": both,
        "event_only": event_only,
        "control_only": control_only,
        "neither_clustered": neither,
        "exact_p_value": exact_mcnemar_pvalue(event_only, control_only)
    })

mcnemar_summary = pd.DataFrame(mcnemar_rows)
display(mcnemar_summary)
mcnemar_summary.to_csv(OUTPUT_DIR / "dbscan_mcnemar_sensitivity_results.csv", index=False)

## 13. Annual stability summary

Annual summaries are calculated from the daily baseline and sensitivity results. This provides a descriptive view of how cluster coverage and noise vary over time. It should not be interpreted as evidence that one configuration detects flood events accurately.

In [ ]:
baseline_daily["date"] = pd.to_datetime(baseline_daily["date"])
baseline_yearly = baseline_daily.groupby(baseline_daily["date"].dt.year).agg(
    baseline_clustered_percent=("clustered_percent", "mean"),
    baseline_noise_percent=("noise_percent", "mean"),
    baseline_clusters=("cluster_count", "mean")
)

selective_daily = sensitivity_daily[
    (sensitivity_daily["eps"] == 1.5) &
    (sensitivity_daily["min_samples"] == 6)
].copy()
selective_yearly = selective_daily.groupby(
    pd.to_datetime(selective_daily["date"]).dt.year
).agg(
    selective_clustered_percent=("clustered_percent", "mean"),
    selective_noise_percent=("noise_percent", "mean"),
    selective_clusters=("cluster_count", "mean")
)

annual_comparison = baseline_yearly.join(selective_yearly).round(2)
display(annual_comparison)
annual_comparison.to_csv(OUTPUT_DIR / "dbscan_final_yearly_comparison.csv")

## 14. Visualize the principal results

The project includes saved figures for the study area, terrain, rainfall, a single-day DBSCAN example, annual configuration comparison and event/control comparison. The code below displays available figures. The maps are visual summaries; the CSV outputs remain the numerical record.

In [ ]:
figure_names = [
    "01_upper_beas_study_area_map.png",
    "02_upper_beas_elevation_map.png",
    "03_upper_beas_slope_map.png",
    "04_upper_beas_rainfall_2015_09_22.png",
    "05_upper_beas_dbscan_clusters_2015_09_22.png",
    "06_dbscan_yearly_configuration_comparison.png",
    "07_dbscan_event_control_comparison.png",
]

from IPython.display import display, Image
for figure_name in figure_names:
    figure_path = FIGURE_DIR / figure_name
    print(figure_name)
    if figure_path.exists():
        display(Image(filename=str(figure_path)))
    else:
        print("  Figure not present in repository yet")

## 15. Results interpretation and limitations

The completed exploratory analysis reported the following headline results:

- The integrated environmental table contains approximately 1.3 million date–grid observations; the DBSCAN modelling table contains approximately 1.276 million observations across 325 grid cells.
- The baseline configuration (`eps=2.0`, `min_samples=4`, environmental weight 0.3) produced high average clustered coverage and relatively few noise points.
- More selective configurations increased the proportion of observations labelled as noise.
- Historical validation was based on a small number of dated and geolocated events. The event/control comparisons did not demonstrate statistically significant discrimination.

### Limitations

1. Only a small subset of the historical inventory has both usable dates and coordinates for event-level validation.
2. CHIRPS and SMAP are gridded satellite/reanalysis products, not in-situ drainage sensors. Their native spatial resolutions are coarser than or comparable to the analysis grid.
3. Daily CHIRPS precipitation cannot resolve sub-daily rainfall intensity.
4. The basin boundary is a provisional HydroBASINS aggregation.
5. DBSCAN results depend on feature scaling, spatial weighting, `eps` and `min_samples`.
6. Clustering indicates density structure, not causality or a calibrated probability of flooding.

Accordingly, the results should be presented as exploratory environmental clustering and not as a validated flash-flood prediction system.

## 16. Reproducibility checklist

Before sharing results or comparing another run, confirm:

- [ ] The input file SHA-256 hashes match `input_data_manifest_sha256.csv`.
- [ ] The same Python, NumPy, pandas, scikit-learn and pyproj versions are used (recorded in the setup cell).
- [ ] Dates are parsed consistently and date–grid records are unique.
- [ ] Missing SMAP observations are not converted to zero.
- [ ] Antecedent rainfall uses the previous days only (`shift(1)`).
- [ ] Soil-moisture features use previous-day values.
- [ ] The scaler is fitted only on observations before 2018-01-01.
- [ ] Spatial coordinates use EPSG:32643 and are divided by 5,000.
- [ ] Environmental features are divided by the fixed weight 0.3.
- [ ] Historical incidents are not included in DBSCAN input features.
- [ ] The same DBSCAN parameter configurations and event/control pairs are used.
- [ ] The `data/README.md` and `docs/data_sources.md` files document the source versions, licenses, preparation steps and exact matching procedure.

### Citation and data acknowledgement

A publication or public repository release should cite CHIRPS, NASA SMAP, SRTM, HydroSHEDS/HydroBASINS and the HiFlo-DAT inventory according to their official documentation and access terms. Include the data-source metadata and the exact dataset versions used in the accompanying `docs/data_sources.md` file.